# Train a model using Autogluon

After preparing the data in the previous step, we read the data, stored in gold, to train a model.

In a real scenario, we would have corrected data for batch effects, which are clearly shown in the previous notebook.

Here we train a simple model to predict healthy/disease, then log it into mlflow.


In [0]:
%pip install -q autogluon
import autogluon

In [0]:
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.metrics import roc_auc_score, confusion_matrix
import mlflow
import mlflow.sklearn

# Step 0: Load sample_group metadata from Delta table
sample_meta = spark.table("silver.methylation.methylation_beta") \
    .select("sample_id", "sample_group") \
    .distinct().toPandas().set_index("sample_id")

# Step 1: Load wide-format methylation matrix from Parquet
df = pd.read_parquet("/dbfs/Volumes/gold/methylation/methylation_features/methylation_beta_001.parquet")
df.set_index("sample_id", inplace=True)

# Step 2: Join sample_group info
df["sample_group"] = sample_meta.loc[df.index, "sample_group"].values

# Step 3: Drop missing target values
df = df.dropna(subset=["sample_group"])
label_col = "sample_group"
X = df.drop(columns=[label_col])
y = df[label_col]

# Step 4: Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

train_data = X_train.copy()
train_data[label_col] = y_train
test_data = X_test.copy()
test_data[label_col] = y_test

# Step 5: Start MLflow tracking

In [0]:

with mlflow.start_run(run_name="AutoGluon_Methylation_Classifier"):
    
    # Step 6: Train with AutoGluon
    predictor = TabularPredictor(label=label_col, eval_metric="roc_auc", verbosity=2).fit(train_data, time_limit=600)

    # Step 7: Evaluate
    y_pred_proba = predictor.predict_proba(test_data)[1]  # get probability of positive class
    y_pred = predictor.predict(test_data)
    y_true = test_data[label_col]

    # Compute metrics
    auroc = roc_auc_score(y_true, y_pred_proba)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=predictor.class_labels).ravel()
    sensitivity = tp / (tp + fn)  # recall
    specificity = tn / (tn + fp)  # specificity

    # Step 8: Log metrics
    mlflow.log_metric("AUROC", auroc)
    mlflow.log_metric("Sensitivity", sensitivity)
    mlflow.log_metric("Specificity", specificity)

    # Log model
    mlflow.sklearn.log_model(predictor, "model")

    print(f"✅ AUROC: {auroc:.3f}, Sensitivity: {sensitivity:.3f}, Specificity: {specificity:.3f}")
